# 11. Custom Workflow Agent — deterministic workflow와 agentic step 섞기

## 학습 목표

- custom workflow가 단일 agent loop와 다른 이유를 이해합니다.
- deterministic validation node와 agentic answer node를 분리합니다.
- LangGraph로 작은 workflow agent를 구성합니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# Observability 설정 — 키가 없으면 비활성 상태로 둡니다.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")
lf_config = {}

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class WorkflowState(TypedDict):
    question: str
    route: str
    answer: str

## 11.1 deterministic router

검증 가능한 규칙은 LLM이 아니라 코드 node에 둡니다.

In [ ]:
def route(state: WorkflowState) -> dict:
    if "sql" in state["question"].lower():
        return {"route": "database"}
    return {"route": "general"}

In [ ]:
def answer(state: WorkflowState) -> dict:
    text = f"{state['route']} workflow로 처리: {state['question']}"
    return {"answer": text}

## 11.2 workflow compile

각 단계가 분리되면 테스트와 관측이 쉬워집니다.

In [ ]:
builder = StateGraph(WorkflowState)
builder.add_node("route", route)
builder.add_node("answer", answer)
builder.add_edge(START, "route")
builder.add_edge("route", "answer")
builder.add_edge("answer", END)
graph = builder.compile()

In [ ]:
graph.invoke({"question": "SQL 결과를 설명해줘", "route": "", "answer": ""})

---

## 정리

| 항목 | 내용 |
|---|---|
| **다룬 기술** | deterministic route, workflow agent, StateGraph |
| **핵심 개념** | custom workflow는 agent 자유도를 낮추는 것이 아니라, 검증 가능한 경계를 만드는 방법입니다. |

**참고 문서:**
- `docs/langchain/multi-agent/custom-workflow.md`
- `docs/langchain/structured-output.md`
- `docs/langgraph/workflows-agents.md`